### 社内文書を想定したテストデータを作成

以下のようなフォルダ構成で、社内文書を想定したテキストファイルを用意します。

```text
C:\python\RAG_test_data
│
├── 人事/
│   ├── 休暇/
│   │   ├── 有給休暇.txt
│   │   ├── 産休.txt
│   │   └── 欠勤.txt
│   │
│   ├── 申請/
│   │   ├── 有給休暇申請書.txt
│   │   └── 休職申請書.txt
│   │
│   └── 評価/
│       ├── 人事評価制度.txt
│       └── 昇格基準.txt
│
├── 労務/
│   ├── 勤怠/
│   │   ├── 勤怠規定.txt
│   │   ├── 遅刻・早退.txt
│   │   └── テレワーク勤務規定.txt
│   │
│   ├── 給与/
│   │   ├── 給与規定.txt
│   │   └── 賞与規定.txt
│   │
│   └── 社会保険/
│       ├── 健康保険.txt
│       └── 厚生年金.txt
│
├── 総務/
│   ├── 福利厚生/
│   │   ├── 福利厚生規定.txt
│   │   └── 慶弔見舞金.txt
│   │
│   └── オフィス/
│       ├── オフィス利用規定.txt
│       └── 会議室利用規定.txt
│
└── 情報システム/
    ├── セキュリティ/
    │   ├── 情報セキュリティ規定.txt
    │   └── パスワード管理.txt
    │
    └── PC/
        ├── PC利用規定.txt
        └── ソフトウェア利用規定.txt
```


In [1]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document

C:\ProgramData\anaconda3\envs\qwen3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Documentを作成する

各テキストファイルをLangChainの `Document` に変換します。

```text
Document
├── page_content
│     └── 文書本文
│
└── metadata
      ├── filename
      └── path
```

`page_content` に文書本文を保存し、`metadata` にはファイル名と元ファイルの相対パスを保存します。


In [2]:
folder_path = Path(r"C:\python\RAG_test_data")

txt_files = list(folder_path.rglob("*.txt"))

#### `rglob()` でサブフォルダを含めてテキストファイルを取得

`rglob()` は、指定したフォルダだけでなく、その下にあるサブフォルダも再帰的に検索します。

```text
RAG_test_data
├── 人事
│   ├── 休暇
│   │   └── 有給休暇.txt
│   └── 申請
│       └── 有給休暇申請書.txt
│
├── 労務
│   └── ...
│
└── 総務
    └── ...
```


In [3]:
print(len(txt_files))

for file_path in txt_files:
    print(file_path)

22
C:\python\RAG_test_data\人事\休暇\有給休暇.txt
C:\python\RAG_test_data\人事\休暇\欠勤.txt
C:\python\RAG_test_data\人事\休暇\産休.txt
C:\python\RAG_test_data\人事\申請\休職申請書.txt
C:\python\RAG_test_data\人事\申請\有給休暇申請書.txt
C:\python\RAG_test_data\人事\評価\人事評価制度.txt
C:\python\RAG_test_data\人事\評価\昇格基準.txt
C:\python\RAG_test_data\労務\勤怠\テレワーク勤務規定.txt
C:\python\RAG_test_data\労務\勤怠\勤怠規定.txt
C:\python\RAG_test_data\労務\勤怠\遅刻・早退.txt
C:\python\RAG_test_data\労務\社会保険\健康保険.txt
C:\python\RAG_test_data\労務\社会保険\厚生年金.txt
C:\python\RAG_test_data\労務\給与\給与規定.txt
C:\python\RAG_test_data\労務\給与\賞与規定.txt
C:\python\RAG_test_data\情報システム\PC\PC利用規定.txt
C:\python\RAG_test_data\情報システム\PC\ソフトウェア利用規定.txt
C:\python\RAG_test_data\情報システム\セキュリティ\パスワード管理.txt
C:\python\RAG_test_data\情報システム\セキュリティ\情報セキュリティ規定.txt
C:\python\RAG_test_data\総務\オフィス\オフィス利用規定.txt
C:\python\RAG_test_data\総務\オフィス\会議室利用規定.txt
C:\python\RAG_test_data\総務\福利厚生\慶弔見舞金.txt
C:\python\RAG_test_data\総務\福利厚生\福利厚生規定.txt


In [4]:
documents = []

for file_path in txt_files:
    # 本文を読み込む
    text = file_path.read_text(encoding="utf-8")
    
    # ルートフォルダからの相対パスを取得する
    relative_path = file_path.relative_to(folder_path)
    
    # 相対パスとファイル名をメタデータとして保存
    metadata = {
        "path": str(relative_path),
        "filename": file_path.name
    }

    # 本文とメタデータからDocumentを作成
    document = Document(
        page_content=text,
        metadata=metadata
    )
    documents.append(document)

#### `documents` の中身を確認する


In [5]:
print("documentsの型を確認:\n")
print(type(documents))
print("\n---------------------------------\n")
print("documentsの要素数を確認:\n")
print(len(documents))
print("\n---------------------------------\n")
print("documents1つ目の本文:\n")
print(documents[0].page_content)
print("\n---------------------------------\n")
print("documents1つ目のメタデータ:\n")
print(documents[0].metadata)

documentsの型を確認:

<class 'list'>

---------------------------------

documentsの要素数を確認:

22

---------------------------------

documents1つ目の本文:

有給休暇規定

1. 有給休暇の概要

有給休暇は、従業員が心身のリフレッシュや私用などのために取得できる休暇です。
会社は、従業員が適切に休暇を取得できるよう、所定の休暇制度を設けています。
有給休暇の取得にあたっては、業務への影響を考慮し、原則として事前に申請してください。

2. 付与日数

有給休暇は入社後10日付与されます。
その後の付与日数は、勤続年数や会社の規定に応じて決定されます。
付与された有給休暇の残日数については、社内の勤怠システムから確認できます。
付与日数や残日数について疑問がある場合は、人事部または労務担当者へ確認してください。

3. 取得方法

有給休暇を取得する場合は、原則として事前に社内の勤怠システムから申請してください。
申請時には取得予定日と取得日数を登録し、所属長の承認を受ける必要があります。
半日単位などの休暇については、会社が定めた取得方法に従って申請してください。

4. 取得日の変更

申請後に取得日を変更する必要が生じた場合は、勤怠システム上の申請内容を確認し、必要な修正を行ってください。
業務上やむを得ない事情がある場合は、所属長と取得時期について相談してください。
休暇取得後は、実際の取得状況と勤怠システムの登録内容に相違がないことを確認してください。

---------------------------------

documents1つ目のメタデータ:

{'path': '人事\\休暇\\有給休暇.txt', 'filename': '有給休暇.txt'}


#### ここまでの処理

```text
TXTファイル
    ↓
本文を読み込む
    ↓
Documentを作成
 ├─ page_content：文書本文
 └─ metadata：ファイル名・保存場所
```

次に、作成した `Document` をチャンクに分割します。


#### Documentをチャンクに分割する


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

#### チャンク分割時の overlap とメタデータ

別の元データ同士が `overlap` でつながることはありません。

```text
有給休暇.txt
   ↓
chunk 0
chunk 1
chunk 2

欠勤.txt
   ↓
chunk 3
chunk 4

有給休暇.txt
chunk 0 ── overlap ── chunk 1 ── overlap ── chunk 2
                                                   │
                                                   │ ← ここで終了
                                                   ↓
```

また、メタデータはそれぞれ元の `Document` から引き継がれます。

```text
A-1 → Aのmetadata
A-2 → Aのmetadata
A-3 → Aのmetadata

B-1 → Bのmetadata
B-2 → Bのmetadata
```


In [7]:
# 有給休暇.txt のチャンクを確認

print(chunks[0].metadata)
print(chunks[0].page_content)
print("\n------------------------\n")
print(chunks[1].metadata)
print(chunks[1].page_content)

{'path': '人事\\休暇\\有給休暇.txt', 'filename': '有給休暇.txt'}
有給休暇規定

1. 有給休暇の概要

有給休暇は、従業員が心身のリフレッシュや私用などのために取得できる休暇です。
会社は、従業員が適切に休暇を取得できるよう、所定の休暇制度を設けています。
有給休暇の取得にあたっては、業務への影響を考慮し、原則として事前に申請してください。

2. 付与日数

有給休暇は入社後10日付与されます。
その後の付与日数は、勤続年数や会社の規定に応じて決定されます。
付与された有給休暇の残日数については、社内の勤怠システムから確認できます。
付与日数や残日数について疑問がある場合は、人事部または労務担当者へ確認してください。

3. 取得方法

有給休暇を取得する場合は、原則として事前に社内の勤怠システムから申請してください。
申請時には取得予定日と取得日数を登録し、所属長の承認を受ける必要があります。
半日単位などの休暇については、会社が定めた取得方法に従って申請してください。

4. 取得日の変更

------------------------

{'path': '人事\\休暇\\有給休暇.txt', 'filename': '有給休暇.txt'}
4. 取得日の変更

申請後に取得日を変更する必要が生じた場合は、勤怠システム上の申請内容を確認し、必要な修正を行ってください。
業務上やむを得ない事情がある場合は、所属長と取得時期について相談してください。
休暇取得後は、実際の取得状況と勤怠システムの登録内容に相違がないことを確認してください。


#### Embeddingモデルを準備してChromaに登録する

今回は多言語Embeddingモデル `intfloat/multilingual-e5-base` を使用します。

1. 各チャンクをベクトル化

```python
vectors = embedding_model.encode(
    [chunk.page_content for chunk in chunks]
)
```

2. Chromaへ登録

```python
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=vectors.tolist(),
    documents=[chunk.page_content for chunk in chunks],
    metadatas=[chunk.metadata for chunk in chunks]
)
```


In [8]:
# Embedding

from sentence_transformers import SentenceTransformer
model_name = "intfloat/multilingual-e5-base"
embedding_model = SentenceTransformer(model_name)

vectors = embedding_model.encode(
    [chunk.page_content for chunk in chunks]
)

print(vectors.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5421.46it/s]


(31, 768)


```
31個のチャンク
   ↓
それぞれをEmbedding
   ↓
31個の768次元ベクトル
```

イメージとして、

```
chunk_0 → [768個の数値]
chunk_1 → [768個の数値]
chunk_2 → [768個の数値]
...
chunk_30 → [768個の数値]
```
となっています。


#### Chromaへ登録するデータ

1つのチャンクに対して、本文・768次元ベクトル・メタデータを対応させて保存します。

```text
chunk_0
├─ 文書本文
├─ 768次元ベクトル
└─ メタデータ
```


#### ChromaのCollectionを作成する

まず、ベクトル・文書本文・メタデータを保存するCollectionを作成します。

```text
ChromaDB
    ↓
C:\python\RAG_test_data\chroma_db に永続化
    ↓
company_documents というCollectionを作成
```

`PersistentClient` でChromaDBへの接続を作り、`get_or_create_collection()` で `company_documents` というCollectionを用意します。

この時点では、チャンク・ベクトル・メタデータを保存するための箱を作成した状態です。

```text
ChromaDB
   │
   └── company_documents
           │
           └── Collection
```


In [9]:
import chromadb

client = chromadb.PersistentClient(
    path=r"C:\python\RAG_test_data\chroma_db"
)

collection = client.get_or_create_collection(
    name="company_documents"
)

print(collection.name)


company_documents


##### Chromaへデータを登録する

`collection.add()` を使って、分割したチャンクをChromaのCollectionへ登録します。

```python
collection.add(
    ids=[...],
    embeddings=vectors.tolist(),
    documents=[...],
    metadatas=[...]
)
```

1つのチャンクに対して、以下の3点を対応させて保存します。

- **Embeddingベクトル**：`embeddings=vectors.tolist()` でChromaに渡せるリスト形式へ変換
- **文書本文（チャンク）**：`page_content` から取得
- **メタデータ**：`metadata` から取得


In [10]:
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=vectors.tolist(),
    documents=[chunk.page_content for chunk in chunks],
    metadatas=[chunk.metadata for chunk in chunks]
)

#### ここまでの処理

```text
ファイル読み込み
    ↓
Document作成
    ↓
チャンク分割
    ↓
Embedding
    ↓
Chromaへ登録
```

次に、登録したデータを検索するための `Retriever` を構築します。


#### Retrieverを構築する

- `collection`：ChromaのCollectionを直接操作するために使用
- `vectorstore`：同じCollectionをLangChainから扱うための窓口

```python
vectorstore = Chroma(
    client=client,
    collection_name="company_documents",
    embedding_function=embedding_function
)
```

先ほど作成した `company_documents` CollectionにLangChainの `Chroma` を接続し、Retrieverとして利用します。

```text
ChromaDB
│
└── company_documents
      │
      ├── chunk_0
      ├── chunk_1
      ├── chunk_2
      └── ...
      ↑
      │
  vectorstore
      │
      ↓
  Retriever
```


In [11]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)

vectorstore = Chroma(
    client=client,
    collection_name="company_documents",
    embedding_function=embedding_function
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5192.49it/s]



#### Retrieverの役割

`retriever` は、ユーザーの質問に近い内容を持つチャンクをChromaから取得するために使用します。

今回は、

```python
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)
```

としているため、質問に対して関連度の高いチャンクを最大3件取得します。

RAGでは、Retrieverが取得した文書がそのまま最終回答になるわけではありません。

```text
ユーザーの質問
    ↓
Retriever
    ↓
関連度の高いチャンクを取得
    ↓
取得したチャンクをContextとしてLLMへ渡す
    ↓
Qwen3が参照情報をもとに回答
```

質問と完全に一致する文書が存在しない場合でも、Retrieverは「比較的近い」チャンクを返すことがあります。
そのため、後段のQwen3側で「参照情報に回答の根拠があるか」を判定させる構成にしています。


### Qwen3を使ってRAGの回答を生成する

取得した参照情報に回答の根拠がない場合は、推測で補完せず「データ上にない情報なのでお答えできません」と返すようにします。


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Loading weights: 100%|██████████| 398/398 [00:02<00:00, 138.82it/s]



#### Qwen3-4Bを回答生成モデルとして使用

Retrieverで取得したチャンクをそのまま表示するのではなく、ローカルLLMである `Qwen3-4B` に渡して自然文の回答を生成します。

```text
Retrieverで取得した文書
        ↓
Contextを作成
        ↓
Qwen3-4Bへ入力
        ↓
社内文書を根拠とした回答
```

今回はローカル環境で動作させるため、Hugging Faceから `Qwen/Qwen3-4B` を読み込んで使用します。

当初は `Qwen/Qwen3-1.7B` を使用していましたが、参照文書に明記されていない情報を推測して回答するケースが確認されました。
そのため、最終的にはモデルサイズを4Bへ変更しています。
詳細は後半の「モデルサイズによる回答品質の比較」で整理します。


#### RAGの回答生成を確認する

Retrieverで取得した文書をContextとしてQwen3へ渡し、回答と出典を表示します。


In [13]:
query = "パソコンの使い方"

# Retrieverで関連するチャンクを取得
results = retriever.invoke(query)

# ============================================================
# Contextを作成
# ============================================================

context = "\n\n".join(
    f"""【資料】
ファイル名: {doc.metadata['filename']}
保存場所: {doc.metadata['path']}

【本文】
{doc.page_content}"""
    for doc in results
)

# ============================================================
# QWENに渡すプロンプト
# ============================================================

prompt = f"""
あなたは社内文書を参照して回答するアシスタントです。

以下の参照情報だけを根拠として回答してください。

参照情報に質問への回答が含まれていない場合は、
「データ上にない情報なのでお答えできません」
と回答してください。

あなた自身の知識や推測で補完してはいけません。

回答は日本語のみで作成してください。
参照情報をそのまま長く転載するのではなく、
質問に必要な情報を整理して回答してください。

回答は途中で文章を切らず、最後まで完結させてください。
また、出典情報はPython側で表示するため、回答本文には出典を書かないでください。

### 参照情報
{context}

### 質問
{query}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

# ============================================================
# QWEN用のChat Template
# ============================================================

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# ============================================================
# トークナイズ
# ============================================================

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# ============================================================
# QWENで生成
# ============================================================

outputs = model.generate(
    **inputs,
    max_new_tokens=1024
)

# 入力部分を除いて回答だけ取得
generated_ids = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)

# ============================================================
# 回答を表示
# ============================================================

print("【回答】")
print(response)

# ============================================================
# 出典を表示
# ============================================================

if "データ上にない情報なのでお答えできません" not in response:

    print("\n【出典】")

    # 同じファイルが複数チャンク取得された場合は重複表示しない
    sources = set()

    for doc in results:
        filename = doc.metadata["filename"]
        path = doc.metadata["path"]

        sources.add((filename, path))

    for filename, path in sources:
        print(f"ファイル名：{filename}")
        print(f"保存場所：{path}")
        print()


【回答】
パソコンの使い方に関する規定は、以下の通りです。

1. 基本的な利用  
会社から貸与されたPCは、原則として業務目的で利用してください。業務に必要な範囲を超えて私的な用途で利用することは避けてください。PCには会社が指定したセキュリティソフトを導入し、必要な更新を適用してください。

2. 持ち出し  
PCを社外へ持ち出す場合は、会社が定めた手続きを確認してください。外出先や自宅などでPCを利用する場合も、第三者から画面を見られないよう注意してください。PCを公共の場所に放置することは禁止されています。

3. 離席時  
離席する場合は画面をロックし、第三者が業務情報を閲覧できないようにしてください。短時間の離席であっても、PCをそのままの状態にしないよう注意してください。

4. 紛失・盗難  
PCを紛失した場合や盗難にあった場合は、速やかに情報システム部へ連絡してください。自分で探し続けるのではなく、会社への報告を優先してください。私的なソフトウェアを無断でインストールすることは禁止されています。

5. ソフトウェア利用  
業務で使用するソフトウェアは、会社が許可したものを利用してください。インターネットから取得したソフトウェアを、許可なく業務用PCへインストールしてはいけません。無料で提供されているソフトウェアであっても、会社の許可が必要となる場合があります。有料ソフトウェアを業務で利用する場合は、事前に会社所定の申請を行ってください。購入前に必要なライセンス数や利用目的などを確認してください。個人で購入したライセンスを会社の業務で使用することは避けてください。

6. セキュリティ  
ソフトウェアのライセンスについても適切に管理する必要があります。退職や異動などによって利用者が変更になった場合は、ライセンスの利用状況を確認してください。利用していないソフトウェアについても、ライセンス上の問題がないか確認してください。

7. オフィス利用  
オフィスを利用する際は、他の従業員の業務を妨げないようにしてください。共用スペースを使用した場合は、利用後に整理整頓を行ってください。会社から貸与された設備や備品は、業務上必要な目的で適切に利用してください。

8. 来訪者  
オフィス内へ外部の人を招く場合は、会社所定のルールに従ってくださ


#### 固定クエリによる初期動作確認

上のセルでは、まず固定した質問を使ってRAG全体の流れを確認しています。

処理は以下の順番です。

1. `retriever.invoke(query)` で関連チャンクを取得
2. 取得したチャンクからContextを作成
3. Contextと質問をプロンプトへ組み込む
4. Chat TemplateでQwen3用の入力形式へ変換
5. トークナイズしてQwen3へ入力
6. 生成結果から回答部分だけを取り出す
7. 回答できた場合は取得元のファイル名と保存場所を表示

この段階では固定クエリを使っていますが、次のセルでは `input()` を使い、任意の質問を入力できる最終形に変更します。



#### Retrieverで取得した文書からContextを作成する

Retrieverから取得した `Document` には、本文である `page_content` と、作成時に保存した `filename`・`path` のメタデータが含まれています。

これらを以下の形式にまとめてQwen3へ渡します。

```text
【資料】
ファイル名: 有給休暇.txt
保存場所: 人事\休暇\有給休暇.txt

【本文】
有給休暇は入社後10日付与されます。
```

複数の検索結果は `"\n\n".join()` で連結し、1つのContextとしてプロンプトへ組み込みます。

この構成にすることで、Qwen3へ文書本文だけでなく「どのファイルから取得した情報か」も同時に渡せます。



#### プロンプトで回答範囲を制御する

RAGでは、検索結果に関連する単語が含まれていても、質問への直接的な答えが書かれているとは限りません。

そのため最終版では、次のルールをプロンプトに明示しています。

- 参照情報に直接的な根拠がある場合のみ回答する
- LLM自身の知識で補完しない
- 推測や推論で結論を作らない
- 根拠がない場合は定型文で回答を拒否する
- 短い・曖昧な質問でも、参照情報に記載された範囲だけで回答する
- 回答本文には出典を書かず、出典表示はPython側で行う

特に、文書にない情報をLLMがもっともらしく補完するハルシネーションを抑えることを重視しています。



#### Chat Templateから回答生成まで

Qwen3では、通常の文字列をそのまま渡すのではなく、`messages` をChat Templateへ変換してモデルへ入力します。

```text
messages
   ↓
tokenizer.apply_chat_template()
   ↓
Qwen3が扱える会話形式の文字列
   ↓
tokenizer()
   ↓
token ID
   ↓
model.generate()
   ↓
生成結果
```

`enable_thinking=False` とすることで、最終回答に内部の思考形式を含めず、回答本文を生成させています。

また、`model.generate()` の出力には入力プロンプト部分も含まれるため、

```python
generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
```

として入力部分を除外し、生成された回答部分だけを取り出しています。



#### 出典表示

回答本文とは別に、Retrieverが取得したDocumentのメタデータから出典を表示します。

```text
【出典】
ファイル名：有給休暇.txt
保存場所：人事\休暇\有給休暇.txt
```

同じファイルから複数チャンクが取得される可能性があるため、`set()` を使って同一ファイルの重複表示を防いでいます。

また、Qwen3が

```text
データ上にない情報なのでお答えできません。
```

と回答した場合には出典を表示しません。

これは、Retrieverが質問と無関係な場合でも「最も近いチャンク」を返す可能性があり、その文書を出典として表示すると利用者に誤解を与えるためです。


In [23]:
query = input("質問を入力してください：")

# Retrieverで関連するチャンクを取得
results = retriever.invoke(query)

# ============================================================
# Contextを作成
# ============================================================

context = "\n\n".join(
    f"""【資料】
ファイル名: {doc.metadata['filename']}
保存場所: {doc.metadata['path']}

【本文】
{doc.page_content}"""
    for doc in results
)

# ============================================================
# QWENに渡すプロンプト
# ============================================================

prompt = f"""
あなたは社内文書を参照して回答するアシスタントです。

以下の参照情報だけを根拠として回答してください。

【重要なルール】
1. 参照情報に質問への直接的な回答が記載されている場合のみ、その情報を根拠として回答してください。
2. 参照情報に記載されていない情報を、あなた自身の知識や推測・推論で補完してはいけません。
3. 「～と考えられます」「～と判断できます」「～の可能性があります」など、参照情報から推測した内容を回答してはいけません。
4. 参照情報に質問への直接的な回答が記載されていない場合は、
「データ上にない情報なのでお答えできません」
と回答してください。
5. 質問が短い、または曖昧な場合でも、参照情報に記載されている範囲で回答してください。
6. 回答は日本語のみで作成してください。
7. 参照情報をそのまま長く転載するのではなく、質問に必要な情報を整理して回答してください。
8. 回答は途中で文章を切らず、最後まで完結させてください。
9. 出典情報はPython側で表示するため、回答本文には出典を書かないでください。

### 参照情報
{context}

### 質問
{query}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

# ============================================================
# QWEN用のChat Template
# ============================================================

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# ============================================================
# トークナイズ
# ============================================================

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# ============================================================
# QWENで生成
# ============================================================

outputs = model.generate(
    **inputs,
    max_new_tokens=1024
)

# 入力部分を除いて回答だけ取得
generated_ids = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)

# ============================================================
# 回答を表示
# ============================================================

print("【回答】")
print(response)

# ============================================================
# 出典を表示
# ============================================================

if "データ上にない情報なのでお答えできません" not in response:

    print("\n【出典】")

    # 同じファイルが複数チャンク取得された場合は重複表示しない
    sources = set()

    for doc in results:
        filename = doc.metadata["filename"]
        path = doc.metadata["path"]

        sources.add((filename, path))

    for filename, path in sources:
        print(f"ファイル名：{filename}")
        print(f"保存場所：{path}")
        print()

質問を入力してください： パソコンが故障した場合の対応方法と、問い合わせ先を教えてください。


【回答】
パソコンが故障した場合の対応方法は、速やかに情報システム部へ連絡することです。問い合わせ先は情報システム部です。

【出典】
ファイル名：会議室利用規定.txt
保存場所：総務\オフィス\会議室利用規定.txt

ファイル名：テレワーク勤務規定.txt
保存場所：労務\勤怠\テレワーク勤務規定.txt

ファイル名：PC利用規定.txt
保存場所：情報システム\PC\PC利用規定.txt




### 任意の質問を入力できる最終版

最終版では、冒頭で質問を固定せず、

```python
query = input("質問を入力してください：")
```

として、Notebook実行時に任意の質問を入力できるようにしました。

これにより、同じRAG構成に対してさまざまな質問を入力し、Retrieverの検索結果・Qwen3の回答・回答拒否・出典表示を確認できます。


### RAGの動作確認

最終版では、質問の具体性や表現を変えながら、Retrieverによる文書検索とQwen3-4Bによる回答生成の動作を確認しました。

#### 1. 文書に明確な回答がある質問

質問：

```text
パソコンが故障した場合はどこに問い合わせればいいですか？
```

実際の出力：

```text
【回答】
パソコンが故障した場合は、情報システム部へ速やかに報告してください。

【出典】
ファイル名：会議室利用規定.txt
保存場所：総務\オフィス\会議室利用規定.txt

ファイル名：テレワーク勤務規定.txt
保存場所：労務\勤怠\テレワーク勤務規定.txt

ファイル名：PC利用規定.txt
保存場所：情報システム\PC\PC利用規定.txt
```

PC利用規定に記載されている内容をもとに、パソコンが故障した場合の問い合わせ先を簡潔に回答できることを確認しました。

---

#### 2. 短く曖昧な質問

質問：

```text
会議室
```

実際の出力：

```text
【回答】
会議室を利用する場合は、社内予約システムから事前に予約してください。予約時には利用日時、利用者、利用目的などを登録してください。長時間の利用が必要な場合は、他の利用者への影響を考慮して必要な時間だけ予約してください。利用時間が終了したら、次の利用者が使用できるように机や椅子などを元の状態へ戻してください。会議中に発生したごみは利用者が処理してください。会議室に私物や資料を残したまま退室しないよう注意してください。設備に問題がある場合は、総務担当者へ連絡してください。

【出典】
ファイル名：会議室利用規定.txt
保存場所：総務\オフィス\会議室利用規定.txt

ファイル名：人事評価制度.txt
保存場所：人事\評価\人事評価制度.txt

ファイル名：オフィス利用規定.txt
保存場所：総務\オフィス\オフィス利用規定.txt
```

「会議室」のような短く具体的な指示を含まない質問でも、会議室利用に関する文書を取得し、参照情報の範囲内で利用方法や注意事項を整理して回答できることを確認しました。

---

#### 3. 言い換えを含む質問

質問：

```text
会議室を使いたい
```

実際の出力：

```text
【回答】
会議室を利用する場合は、社内予約システムから事前に予約してください。予約時には利用日時、利用者、利用目的などを登録してください。

【出典】
ファイル名：会議室利用規定.txt
保存場所：総務\オフィス\会議室利用規定.txt

ファイル名：オフィス利用規定.txt
保存場所：総務\オフィス\オフィス利用規定.txt

ファイル名：PC利用規定.txt
保存場所：情報システム\PC\PC利用規定.txt
```

文書では「会議室を利用する」と記載されていますが、「会議室を使いたい」という異なる表現でも関連文書を検索し、会議室の利用方法を回答できることを確認しました。

---

#### 4. 文書に存在しない情報

質問：

```text
日本で一番高い山は？
```

実際の出力：

```text
【回答】
データ上にない情報なのでお答えできません。
```

Retrieverは質問に対して関連度の高いチャンクを取得しますが、取得した文書に回答の根拠が存在しない場合は、Qwen3-4Bが自身の知識で補完せず回答を拒否できることを確認しました。

また、回答できない場合にはRetrieverが取得した無関係な文書を出典として表示しないようにしています。

---

#### 検証結果について

以上の検証から、明確な質問だけでなく、短い質問や言い換えを含む質問についても関連文書を検索し、参照情報をもとに回答を生成できることを確認しました。

また、参照文書に回答の根拠が存在しない質問については、LLM自身の知識による補完を行わず、回答を拒否する動作も確認できました。

一方で、Retrieverは関連度の高い文書を複数取得するため、回答に直接使用していない文書が出典候補として表示される場合があります。今後の改善点として、実際に回答生成の根拠として使用した文書のみを出典として表示する仕組みを検討します。



### モデルサイズによる回答品質の比較

当初は `Qwen/Qwen3-1.7B` を使用していました。

しかし、次の質問で問題が発生しました。

```text
有給休暇を取得すると給与は減りますか？
```

参照した勤怠規定には「勤怠情報は給与計算にも利用される」と記載されていますが、
「有給休暇を取得すると給与が減る・減らない」という直接的な記載はありません。

それにもかかわらず、1.7Bでは「給与は減らない」と推測した回答を生成しました。

そこで、

1. プロンプトに「直接的な根拠がある場合のみ回答する」
2. 「推測・推論で補完しない」
3. 「根拠がなければ定型文で回答拒否する」

という制約を追加しました。

それでも1.7Bでは推測した回答が残ったため、モデルを `Qwen/Qwen3-4B` に変更しました。

4Bでは同じ質問に対して、

```text
データ上にない情報なのでお答えできません。
```

と回答し、参照文書に明記されていない情報を回答しない動作を確認できました。

今回の検証から、RAGの回答品質はRetrieverやプロンプトだけでなく、回答生成に使用するLLMの性能にも影響されることを確認しました。



### 今回のRAGで確認できたこと

今回の実装では、次の動作を確認しました。

- フォルダ階層を持つ社内文書をDocumentとして読み込める
- ファイル名・保存場所をメタデータとして保持できる
- 文書をチャンク分割してEmbeddingし、Chromaへ保存できる
- Retrieverで質問に関連するチャンクを取得できる
- Qwen3が検索結果だけを参照して回答を生成できる
- 文書に直接的な根拠がない質問への回答を拒否できる
- 短い質問や言い換えにも対応できる
- 複数文書の情報をまとめて回答できる
- 回答に使用した文書の出典を表示できる
- 回答できない場合は無関係な出典を表示しない
- Qwen3-1.7Bと4Bで、プロンプト遵守と回答品質に差があることを確認できた

単純に検索結果をLLMへ渡すだけでなく、実際の質問パターンを変えながら回答精度とハルシネーションの有無を確認し、モデルとプロンプトを調整しました。
